In [19]:
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import chromadb
import requests
import os
from groq import Groq
import certifi
import json
import time
from collections import deque


In [3]:
import sys
print(sys.executable)
print(sys.version)

load_dotenv()

# Load project API keys from .env and expose both standard and project-specific names.
for env_name in ["GROQ_API_KEY", "groq_api_key", "SAPI_KEY", "sapi_key", "NCBI_API_KEY", "pubapi_key"]:
    value = os.getenv(env_name)
    if value:
        os.environ.setdefault(env_name.upper(), value)
        os.environ.setdefault(env_name.lower(), value)

api_key = os.getenv("NCBI_API_KEY") or os.getenv("pubapi_key")
if api_key:
    os.environ["NCBI_API_KEY"] = api_key
    os.environ["pubapi_key"] = api_key

# Also set the common name used by Groq
if os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


c:\Users\jesus\anaconda3\envs\rag_build\python.exe
3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]

  Attempting uninstall: groq

    Found existing installation: groq 1.7.0

    Uninstalling groq-1.7.0:

      Successfully uninstalled groq-1.7.0

   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 2/2 [langchain-groq]



In [4]:
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

In [8]:
from dotenv import load_dotenv

In [9]:
def fetch_papers(query, total, batch_size=20, max_retries=8):
    papers = []
    offset = 0
    url = "https://api.semanticscholar.org/graph/v1/paper/search"

    while len(papers) < total:
        remaining = total - len(papers)
        page_size = min(batch_size, remaining)

        params = {
            "query": query,
            "limit": page_size,
            "offset": offset,
            "fields": "title,abstract,url,publicationDate,authors,year",
            "year": "2024-",
        }

        headers = {
            "x-api-key": os.environ.get("SAPI_KEY") or os.environ.get("sapi_key"),
            "User-Agent": "Mozilla/5.0",
            "Accept": "application/json"
        }

        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=headers, timeout=30)
            except requests.exceptions.RequestException as e:
                print(f"Request error: {e}")
                time.sleep(5)
                continue

            if response.status_code == 200:
                batch = response.json().get("data", [])
                if not batch:
                    print("No more papers found.")
                    return papers[:total]

                papers.extend(batch)
                offset += len(batch)
                print(f"Fetched {len(papers)} papers so far...")
                time.sleep(2)
                break

            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 2 ** attempt))
                print(f"Rate limited. Waiting {wait} seconds before retrying...")
                time.sleep(wait)
                continue

            print(f"Error {response.status_code}: {response.text}")
            return papers[:total]

        else:
            print("Max retries reached. Stopping early.")
            break

    return papers[:total]

if __name__ == "__main__":
    results1 = fetch_papers("artificial intelligence in neuroscience", total=200, batch_size=10)
    with open("sementicscholar_paper.json", "w") as f:
        json.dump(results1, f, indent=2)

Rate limited. Waiting 1 seconds before retrying...
Rate limited. Waiting 2 seconds before retrying...
Rate limited. Waiting 4 seconds before retrying...
Rate limited. Waiting 8 seconds before retrying...


KeyboardInterrupt: 

In [6]:
import arxiv

def fetch_arxiv_papers(query, total):
    papers = []

    client = arxiv.Client(page_size=200, delay_seconds=3, num_retries=3)

    search = arxiv.Search(
        query=query,
        max_results=total,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )

    for r in client.results(search):
        papers.append({
            "title": r.title,
            "abstract": r.summary,
            "url": r.entry_id,
            "published" : str(r.published),
            "authors": [a.name for a in r.authors],
        })
    return papers

if __name__ == "__main__":
    results2 = fetch_arxiv_papers(
        query= "(cat:cs.AI or cat:cs.LG or cat:cs.NE) and (abs:neuroscience OR abs:brain or ti:neuroscience)",
        total=200
    )
    
    with open("arxiv_paper.json", "w") as f:
        json.dump(results2, f, indent=2)

In [10]:
from metapub import PubMedFetcher

fetch = PubMedFetcher()


def fetch_pubmed_papers(query, total, max_retries=5):
    papers = []
    api_key = os.environ.get("NCBI_API_KEY") or os.environ.get("pubapi_key")

    if not api_key:
        print("No PubMed API key found in environment. Requests may be rate-limited.")

    pmids = fetch.pmids_for_query(
        query,
        retmax=min(total, 50),
        since="2024/01/01",
        api=api_key,
    )

    for pmid in pmids:
        for attempt in range(max_retries):
            try:
                r = fetch.article_by_pmid(pmid)
                papers.append({
                    "title": r.title,
                    "abstract": r.abstract,
                    "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                    "published": str(r.year),
                    "authors": r.authors,
                })
                time.sleep(10)
                break
            except Exception as e:
                print(f"Error fetching paper with PMID {pmid} (attempt {attempt + 1}/{max_retries}): {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
                    continue
                break
    return papers


if __name__ == "__main__":
    results3 = fetch_pubmed_papers(
        query="(artificial intelligence OR machine learning) AND (neuroscience OR brain)",
        total=200,
    )

    with open("pubmed_paper.json", "w") as f:
        json.dump(results3, f, indent=2)


KeyboardInterrupt: 

In [ ]:
import re 
import hashlib

def merge_json_file(file_paths):

    merged_data = []
    for path in file_paths:
        with open(path, "r") as f:
            data = json.load(f)
            merged_data.extend(data)

    with open("merged_papers.json", "w") as f:
        json.dump(merged_data, f, indent=2)

    return merged_data

def clean_paper_text(text):
    if not text:
        return ""
    text = re.sub(r'References.*?(?=\n[A-Z]|', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'(Author|Corresponding|Affiliation|Conflict of Interest).?\n', '', text, re.IGNORECASE)

    return text.strip()

def deduplicate_papers(papers):
    seen = set()
    unique = []

    for paper in papers:
        if paper["title"] not in seen:
            seen.add(paper["title"])
            unique.append(paper)

    return unique

def chunk_abstracts(papers: list) -> list:
    chunks = []

    for paper in papers:
        abstract = clean_paper_text(paper.get('abstract' , ''))

        if not abstract:
            continue

        chunk = {
            "text" : f"Title: {paper['title']}\n\n{abstract}",
            "meta_data" : {
                "source" : paper.get("title", "Unknown"),
                "url" : paper.get("url", ""),
                "published" : paper.get("published", ""),
            }
        }
        source = chunk["meta_data"]["source"]
        url = chunk["meta_data"]["url"]
        hash_input = f"{source}_{url}".encode('utf-8')
        chunk['id'] = hashlib.md5(hash_input).hexdigest()[:12]

        chunks.append(chunk)

    return chunks

with open("merged_papers.json", "r") as f:
    merged_data = json.load(f)

pre_chunks = deduplicate_papers(merged_data)

chunked = chunk_abstracts(pre_chunks)

print(f"Created {len(chunked)} chunks")

#save 
with open("chunked.json", "w") as f:
    json.dump(chunked, f, indent=2)

In [30]:
with open("chunked.json", "r") as f:
    chunks = json.load(f)

In [ ]:
import torch 
from sentence_transformers import SentenceTransformer
import warnings

warnings.filterwarnings("ignore")

model_name = "allenai/scibert_scivocab_uncased"

model = SentenceTransformer("sentence-transformer/allenai-specter")

In [ ]:
texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(texts, convert_to_tensor=True , show_progress_bar=True) #check tensor 

In [ ]:
with open("text_of_chunks.json", 'w') as f:
    json.dump(texts,f, indent=2)

with open("embeddings.json", "w") as f:
    json.dump(embeddings.tolist(), f, indent=2)

In [31]:
with open("embeddings.json", "r") as f:
    embeddings = json.load(f)


In [32]:
import chromadb

client = chromadb.PersistentClient(path="./my_chroma_db")

collection = client.get_or_create_collection(name="documents", embedding_function=None) #

ids = [chunk["id"] for chunk in chunks]
documents = [chunk['text'] for chunk in chunks]
metadatas = [{'source': chunk['meta_data']['source'], 
              'url': chunk['meta_data']['url'], 
              'published': chunk['meta_data']['published']} for chunk in chunks]

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings
)

print(f"Added {len(chunks)} documents to the ChromaDB collection.")

Added 414 documents to the ChromaDB collection.


In [21]:
from langchain_groq import ChatGroq

load_dotenv()

GroqClient = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [35]:
def relevant_chunks(user_query, top_k=5):
    query_embedding = model.encode([user_query], convert_to_tensor=True)
    query_embedding = query_embedding.detach().cpu().tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    return results['documents'][0], results['metadatas'][0]

In [23]:
def maintain_context(response, context_history, max_context_length=10):
    if len(context_history) >= max_context_length:
        context_history.popleft()  

    context_history.append(response)
    return context_history

In [38]:
system_message = """You are a medical AI research assistant specializing in neuroscience and medical imaging.

Critical Rules:
- Answer ONLY using information from the provided papers
- Do NOT use knowledge outside the context
- If information is not found, clearly state: "This question is not addressed in the provided papers."
- When you cite information, reference the source paper by title
- If papers contain conflicting claims, note both perspectives and indicate which is more strongly supported"""

prompt_template = """Based ONLY on the provided abstracts of research papers, answer this question:

Question: {question}

Abstracts from papers:
{context}

Your answer (must cite sources):"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("human", prompt_template)
])


def generate_response2(context, user_query, context_history=None, modelLLM="openai/gpt-oss-120b"):
    if isinstance(context, tuple):
        context, metadatas = context
        metadatas = json.dumps(metadatas)
    else:
        metadatas = ""

    if isinstance(context, list):
        context = "\n\n".join(str(document) for document in context)
    else:
        context = str(context)

    if context_history:
        context = " ".join(context_history) + "\n\n" + context + "\n\n" + metadatas

    llm = ChatGroq(
        model=modelLLM,
        streaming=True
    )

    chain = prompt | llm

    print(">> groq: ", end="", flush=True)
    full_response = ""

    for chunk in chain.stream({"context": context, "question": user_query}):
        token = chunk.content if hasattr(chunk, "content") else ""
        if isinstance(token, str) and token:
            print(token, end="", flush=True)
            full_response += token

    print("\n")
    return full_response

In [25]:
def rag_pipeline(user_query, context_history):
    relevant_chunks_query = relevant_chunks(user_query, top_k=5)
    response = generate_response2(relevant_chunks_query, user_query, context_history)
    
    return response

In [26]:
def talk_to_groq():
    context_history = deque()
    while True:
        user_input = input(">> you: ")
        if user_input.lower() == "bye":
            print("Exiting the conversation. Goodbye!")
            break
        response = rag_pipeline(user_input, context_history)
        context_history = maintain_context(response, context_history)

In [40]:
def evaluation():
    with open("evaluation.json", "r", encoding="utf-8") as f:
        evaluation_data = json.load(f)

    if isinstance(evaluation_data, dict):
        questions_list = evaluation_data.get("questions", [])
    else:
        questions_list = evaluation_data

    evaluation_results = []

    for item in questions_list:
        question = item.get("question", "")
        reference_answer = item.get("expected_answer", "")
        expected_answerable = item.get("answerable")
        relevant_papers = set(item.get("relevant_papers", []))

        if not question:
            continue

        try:
            retrieved_context = relevant_chunks(question, top_k=5)
            answer = generate_response2(retrieved_context, question, deque())

            retrieved_sources = retrieved_context[1]
            retrieved_titles = {source.get("source", "") for source in retrieved_sources}
            source_recall = (
                len(relevant_papers & retrieved_titles) / len(relevant_papers)
                if relevant_papers else 0
            )

            context_text = "\n\n".join(retrieved_context[0])
            judge_prompt = f"""
Evaluate this RAG response using only the provided context.

Question:
{question}

Context:
{context_text}

RAG response:
{answer}

Expected answer:
{reference_answer}

The evaluation set says this question is answerable from the corpus:
{expected_answerable}

For an unanswerable question, a good response must clearly abstain rather than invent an answer.
Return valid JSON with integer scores from 1 to 5:
{{
  "faithfulness": 1,
  "relevance": 1,
  "citation_quality": 1,
  "answerability_handling": 1,
  "overall": 1,
  "reason": "brief explanation"
}}
"""

            judge_response = GroqClient.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {
                        "role": "system",
                        "content": "You are a strict evaluator of retrieval-augmented generation systems."
                    },
                    {"role": "user", "content": judge_prompt},
                ],
                temperature=0,
                response_format={"type": "json_object"},
            )

            scores = json.loads(judge_response.choices[0].message.content)

            evaluation_results.append({
                "id": item.get("id"),
                "question": question,
                "expected_answerable": expected_answerable,
                "answer": answer,
                "retrieved_sources": retrieved_sources,
                "source_recall": source_recall,
                "scores": scores,
            })

            print(f"Evaluated {item.get('id')}: {question}")

        except Exception as e:
            evaluation_results.append({
                "id": item.get("id"),
                "question": question,
                "expected_answerable": expected_answerable,
                "error": str(e),
            })
            print(f"Evaluation failed for {item.get('id')}: {e}")

    valid_results = [
        result for result in evaluation_results
        if "scores" in result
    ]

    def average(score_name):
        return (
            sum(result["scores"].get(score_name, 0) for result in valid_results)
            / len(valid_results)
            if valid_results else 0
        )

    summary = {
        "num_questions": len(evaluation_results),
        "num_successful": len(valid_results),
        "average_faithfulness": average("faithfulness"),
        "average_relevance": average("relevance"),
        "average_citation_quality": average("citation_quality"),
        "average_answerability_handling": average("answerability_handling"),
        "average_overall": average("overall"),
        "average_source_recall": (
            sum(result["source_recall"] for result in valid_results)
            / len(valid_results)
            if valid_results else 0
        ),
    }

    output = {
        "summary": summary,
        "results": evaluation_results,
    }

    with open("rag_evaluation_results.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(json.dumps(summary, indent=2))
    return output

In [41]:
results = evaluation()

>> groq: The abstracts that discuss concrete uses of AI within neuroscience point to a relatively narrow set of “major application areas.”  In particular, the review titled **“Artificial intelligence in neuroscience: opportunities and prospects: A review”** enumerates the domains in which AI methods are already being deployed:

* **Clinical diagnosis and prognosis of neurological diseases** – AI‑based tools are being used to detect and classify a range of conditions, including  
  * **Stroke**  
  * **Traumatic brain injury (TBI)**  
  * **Neurodegenerative disorders** such as **Parkinson’s disease**, **Alzheimer’s disease**, and **multiple sclerosis**  
  * **Epilepsy**  
  * **Sleep‑related disorders**  

These disease‑focused applications dominate the bibliometric picture of AI in neuroscience that the provided literature highlights.  

In addition, two editorial‑type abstracts emphasize broader, research‑oriented uses of AI in neuroscience:

* Translating AI techniques to **cogniti

In [39]:
if __name__ == "__main__":
    talk_to_groq()

>> groq: **Correlation between neuroscience and computer science (as reflected in the abstracts)**  

1. **AI as a methodological engine for neuroscience**  
   - The abstract *“Artificial intelligence is reshaping neuroscience across scales.”* states that artificial‑intelligence techniques are now “a driver of measurement and, increasingly, a generative engine for hypotheses” at every level of brain research—from protein structure prediction to population‑level dynamics and biomechanics. This directly links computer‑science methods (machine learning, simulation‑based inference, computer vision, etc.) to core neuroscientific questions.  

2. **Big‑data and large‑language models bridging disciplines**  
   - In the interview *“Danilo Bzdok.”* the author argues that AI and big‑data “herald a paradigm shift in neuroscience,” allowing researchers to move beyond intuition‑driven taxonomies and to integrate fragmented knowledge across disciplines. Large language models, a product of computer

KeyboardInterrupt: Interrupted by user